# SaaS Churn Predictor Analysis
## Milestone 4: Churn Predictors (Python + Distributions)

**Author:** Anuj Saini  
**Date:** July 7, 2026  

This notebook performs customer-level behavioral feature engineering using event log activity and support ticket records. It compares the distributions of these features between active (retained) and churned customers to identify the strongest indicators of customer churn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from dotenv import load_dotenv
import os

# Configure style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (18, 6)

# Load database environment
load_dotenv(dotenv_path="../.env")

### 1. Load Data

We retrieve the companies, subscriptions, events, and support tickets from the database.

In [ ]:
host = os.getenv("host")
port = os.getenv("port")
database = os.getenv("database")
username = os.getenv("username")
password = os.getenv("password")

conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=username,
    password=password
)

df_companies = pd.read_sql("SELECT id FROM saas.legacy_companies;", conn)
df_subs = pd.read_sql("SELECT id, company_id, plan, start_date, end_date, status FROM saas.legacy_subscriptions;", conn)
df_events = pd.read_sql("SELECT id, company_id, event_type, event_date FROM saas.legacy_events;", conn)
df_tickets = pd.read_sql("SELECT id, company_id, priority, category FROM saas.legacy_support_tickets;", conn)
conn.close()

df_subs['start_date'] = pd.to_datetime(df_subs['start_date'])
df_subs['end_date'] = pd.to_datetime(df_subs['end_date'])
df_events['event_date'] = pd.to_datetime(df_events['event_date'])

print(f"Loaded {df_companies.shape[0]} companies, {df_subs.shape[0]} subscriptions, {df_events.shape[0]} events, and {df_tickets.shape[0]} tickets.")

### 2. Determine Customer Status and Reference Dates

- Retained: The latest subscription has `status` in `['active', 'paused', 'trial']` (as of the end of the dataset, April 7, 2026).
- Churned: The latest subscription has `status = 'churned'` (with cancellation date `end_date` as the reference date).

In [ ]:
dataset_end_date = df_events['event_date'].max()

# Sort subscriptions to get the latest
df_subs_sorted = df_subs.sort_values(['company_id', 'start_date', 'id'], ascending=[True, False, False])
df_latest_sub = df_subs_sorted.groupby('company_id').first().reset_index()

company_status = []
for idx, row in df_companies.iterrows():
    comp_id = row['id']
    sub_row = df_latest_sub[df_latest_sub['company_id'] == comp_id].iloc[0]
    
    status = 'churned' if sub_row['status'] == 'churned' else 'retained'
    ref_date = sub_row['end_date'] if status == 'churned' else dataset_end_date
    if pd.isna(ref_date):
        ref_date = dataset_end_date
        
    company_status.append({
        'company_id': comp_id,
        'status': status,
        'ref_date': ref_date
    })
    
df_status = pd.DataFrame(company_status)
print(df_status['status'].value_counts())

### 3. Feature Engineering

We calculate behavioral features for each company prior to their reference date.

In [ ]:
feature_records = []
for idx, row in df_status.iterrows():
    comp_id = row['company_id']
    status = row['status']
    ref_date = row['ref_date']
    
    c_events = df_events[(df_events['company_id'] == comp_id) & (df_events['event_date'] <= ref_date)]
    c_tickets = df_tickets[df_tickets['company_id'] == comp_id]
    
    ticket_count = len(c_tickets)
    critical_tickets = len(c_tickets[c_tickets['priority'].isin(['critical', 'high'])])
    
    total_events = len(c_events)
    breadth = c_events['event_type'].nunique()
    
    l30_start = ref_date - pd.Timedelta(days=30)
    events_l30 = len(c_events[c_events['event_date'] >= l30_start])
    logins_l30 = len(c_events[(c_events['event_date'] >= l30_start) & (c_events['event_type'] == 'login')])
    
    exports = len(c_events[c_events['event_type'] == 'export'])
    comments = len(c_events[c_events['event_type'] == 'comment_created'])
    
    feature_records.append({
        'company_id': comp_id,
        'status': status,
        'ticket_count': ticket_count,
        'critical_tickets': critical_tickets,
        'total_events': total_events,
        'breadth': breadth,
        'events_l30': events_l30,
        'logins_l30': logins_l30,
        'exports': exports,
        'comments': comments
    })

df_features = pd.DataFrame(feature_records)
df_features.head()

### 4. Descriptive Statistics & Gap Quantification

We aggregate features by status to look at differences.

In [ ]:
comparison = df_features.groupby('status').agg({
    'company_id': 'count',
    'ticket_count': 'mean',
    'critical_tickets': 'mean',
    'total_events': 'mean',
    'breadth': 'mean',
    'events_l30': 'mean',
    'logins_l30': 'mean',
    'exports': 'mean',
    'comments': 'mean'
}).rename(columns={'company_id': 'count'})

retained = comparison.loc['retained']
churned = comparison.loc['churned']

print("Percentage differences (Retained vs Churned):")
for col in comparison.columns[1:]:
    ret_val = retained[col]
    chu_val = churned[col]
    pct_diff = ((ret_val - chu_val) / chu_val * 100) if chu_val > 0 else 0
    print(f"{col:18s}: Retained={ret_val:.2f} | Churned={chu_val:.2f} | Diff={pct_diff:+.1f}%")

comparison.T

### 5. Distribution Comparison Visualizations

We render boxplots for our top three identified signals: Critical/High Support Tickets, Comment Created volume, and Export volume.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1
sns.boxplot(ax=axes[0], data=df_features, x='status', y='critical_tickets', hue='status',
            palette={'retained': '#2ca02c', 'churned': '#d62728'}, legend=False)
axes[0].set_title('Critical/High Tickets per Customer', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Customer Status')
axes[0].set_ylabel('Ticket Count')

# Plot 2
sns.boxplot(ax=axes[1], data=df_features, x='status', y='comments', hue='status',
            palette={'retained': '#2ca02c', 'churned': '#d62728'}, legend=False)
axes[1].set_title('Comment Created Events per Customer', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Customer Status')
axes[1].set_ylabel('Event Count')

# Plot 3
sns.boxplot(ax=axes[2], data=df_features, x='status', y='exports', hue='status',
            palette={'retained': '#2ca02c', 'churned': '#d62728'}, legend=False)
axes[2].set_title('Data Export Events per Customer', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Customer Status')
axes[2].set_ylabel('Event Count')

plt.suptitle('Behavioral Feature Distributions: Retained vs Churned Customers', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()